# 03 — Out-of-bag estimation

*Chapter 06 — Random Forests · Notebook 3 of 5*

In notebook 1 we noticed something and set it aside: every bootstrap leaves about **37%** of the
training points unseen, and we promised they would "earn their keep in notebook 3." This is that
notebook. Those left-out points turn out to be a **free validation set** — the forest can grade
itself, honestly, without holding back a single row for a separate split.

**Prerequisites.**
- **Notebook 1**: the bootstrap (sampling with replacement) and the ~37% it leaves out.
- **Notebook 2**: the full random forest (bagging + feature subsampling).
- **Module 00**: the train/test split (NB 04) and cross-validation (NB 10) — the honest-evaluation
  discipline OOB joins.

**What you'll be able to do.**
- Derive the ~37% **out-of-bag** fraction from `(1 − 1/n)ⁿ → 1/e`, and measure it.
- Build the **OOB prediction** for each point by hand, and read off the **OOB error**.
- Match your hand-built OOB to scikit-learn's `oob_score_`.
- Use OOB as a free estimate of generalization — and say when it is trustworthy and when it is not.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()
SEED = 0

data = load_breast_cancer()
X, y = data.data, (data.target == 0).astype(int)  # malignant = 1
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
n = X_train.shape[0]
print(f"train: {n} points    test: {X_test.shape[0]} points")

## The points the bootstrap left behind

A bootstrap sample draws n points with replacement, so on average it includes only about 63% of the
distinct points and **leaves ~37% out** (notebook 1). We have been treating those left-out points as a
curiosity. They are in fact a gift: for the tree trained on that bootstrap, they are data it **never
saw** — a ready-made held-out set. With a whole forest of bootstraps, every point is left out by many
trees, and those trees can judge it fairly.

## The bootstrap's leftover is a free test set

Think of one tree. It trained on its bootstrap — about 63% of the points — and the other ~37% are,
*for it*, unseen test data. (Why 37%? As in notebook 1: a point is missed by one draw with probability
1 − 1/n, and the bootstrap draws n times, so it is missed by **all** of them with probability
(1 − 1/n)ⁿ, which tends to 1/e ≈ 0.37 as n grows.)

Now look across the forest: each training point is left out by roughly 37% of the trees, so each point
has its own little committee of trees that **never trained on it**. Ask only those trees to predict it
and you get an honest, out-of-sample guess — for every point, at no extra cost. Pool the errors and you
have the **out-of-bag (OOB) error**: a validation estimate baked into the single fit that built the
forest.

In [ ]:
# The left-out fraction, from the formula and from a direct count.
print(f"(1 - 1/n)**n = {(1 - 1 / n) ** n:.4f}    1/e = {1 / np.e:.4f}    (n = {n})")

rng = np.random.default_rng(SEED)
left_out = [(n - len(np.unique(rng.integers(0, n, n)))) / n for _ in range(3000)]
print(f"empirical mean left-out fraction over 3000 bootstraps: {np.mean(left_out):.4f}")

**Read the result.** The left-out fraction is a robust **~37%** — and at n = 398 it already sits on its
limit, `1/e`. (Notebook 1's n = 10 demo gave 0.349; the larger the sample, the closer to 0.368.) So in a
forest of B trees, each point is out-of-bag for about 0.37·B of them — a real committee, once B is more
than a handful.

## Building the OOB prediction

The recipe is direct. For each training point, gather the trees whose bootstrap did **not** include it,
take their **majority vote**, and call that the point's **OOB prediction** — a prediction from trees
that never saw it. Do this for every point and compare to the truth: the error rate is the **OOB
error**. Notice what we did *not* do: we never held back a validation split. The held-out data was
hiding inside the bootstraps all along.

In [ ]:
# A bag of 200 trees; for each point, vote only the trees whose bootstrap missed it.
n_trees = 200
rng = np.random.default_rng(SEED)
oob_votes = np.zeros((n, 2))        # per-point tally of class votes, from trees that did NOT see it
oob_graders = np.zeros(n)           # how many trees graded each point

for b in range(n_trees):
    idx = rng.integers(0, n, n)     # a bootstrap sample (with replacement)
    tree = DecisionTreeClassifier(max_features="sqrt", random_state=b)
    tree.fit(X_train[idx], y_train[idx])
    oob = np.ones(n, dtype=bool)
    oob[np.unique(idx)] = False     # the points this tree never saw
    preds = tree.predict(X_train[oob]).astype(int)
    np.add.at(oob_votes, (np.where(oob)[0], preds), 1)  # scatter-add a vote per OOB point
    oob_graders[oob] += 1

graded = oob_graders > 0
oob_prediction = oob_votes.argmax(axis=1)
hand_oob_acc = accuracy_score(y_train[graded], oob_prediction[graded])
print(f"hand OOB accuracy: {hand_oob_acc:.4f}")
print(f"points graded: {graded.sum()}/{n}   mean OOB graders/point: {oob_graders.mean():.1f}")

**Read the result.** Every one of the 398 training points was graded by a committee of trees that never
trained on it — about **73 trees each** (≈ 0.37 × 200), exactly as the fraction predicted. The pooled
accuracy, **0.962**, is an out-of-sample estimate we got *for free*, during the same fit that built the
forest. No second split, no refitting.

In [ ]:
# Which points each of a few trees saw (in-bag) vs missed (out-of-bag).
demo_trees, demo_points = 8, 25
demo_rng = np.random.default_rng(SEED)
inbag = np.zeros((demo_trees, demo_points))
for t in range(demo_trees):
    inbag[t, np.unique(demo_rng.integers(0, demo_points, demo_points))] = 1  # 1 = in-bag, 0 = OOB

fig, ax = plt.subplots(figsize=(9, 4))
ax.imshow(inbag, cmap=ListedColormap([COLORS["highlight"], COLORS["grid"]]), aspect="auto")
ax.set_xlabel("training point")
ax.set_ylabel("tree")
ax.set_yticks(range(demo_trees))
ax.set_title("in-bag (grey) vs out-of-bag (coloured) — each tree misses ~37%")
ax.grid(False)
fig.tight_layout()
print(f"OOB cells in this demo grid: {1 - inbag.mean():.0%} (expected ~37%)")

**Read the figure.** Each row is one tree; the grey cells are the points it trained on, the coloured
cells the ~37% it missed. Read it two ways. Along a **row**: every tree sits out about a third of the
data. Down a **column** (one point): the coloured cells are exactly the trees allowed to grade it — the
ones that never saw it. A point never votes on a tree that trained on it, and that is the whole reason
the estimate is honest.

In [ ]:
forest = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=SEED)
forest.fit(X_train, y_train)
print(f"scikit-learn oob_score_: {forest.oob_score_:.4f}")
print(f"our hand-built OOB:      {hand_oob_acc:.4f}")
print(f"sealed test accuracy:    {accuracy_score(y_test, forest.predict(X_test)):.4f}")

**Read the parity — and the caveat.** scikit-learn's `oob_score_` (0.955) lands right next to our
hand-built OOB (0.962); the library is doing exactly what we did, point by point. The two differ only by
a point because they draw **different random bootstraps** — the same OOB estimator, a little RNG wobble
on a 398-point training set (resample and either can land on top). And the OOB estimate sits close to
the **sealed test** (0.942) — but a touch *above* it. OOB is honest in spirit yet mildly **optimistic**:
it is still measured on the training points. Use it as a cheap running estimate; report the **sealed
test** as the final number.

## When is OOB trustworthy?

There is a catch, and it is about *how many* trees. A point is graded only by the trees that happened to
miss it — about 37% of them. With a healthy forest that is plenty. But with **few** trees, some points
are unlucky and land in-bag for *every* tree, so they have **no** OOB grader at all, and the estimate is
built from a shrinking, biased subset. scikit-learn does not hide this — it raises a warning when too
few trees leave points ungraded. Let's watch the estimate settle as the forest grows.

In [ ]:
tree_counts = [3, 5, 10, 25, 50, 100, 200, 500]
oob_err, test_err = [], []
for B in tree_counts:
    m = RandomForestClassifier(n_estimators=B, oob_score=True, random_state=SEED)
    m.fit(X_train, y_train)  # sklearn WARNS at small B (some points never OOB) — we let it through
    oob_err.append(1 - m.oob_score_)
    test_err.append(1 - accuracy_score(y_test, m.predict(X_test)))

fig = viz.plot_train_test_curve(
    tree_counts, oob_err, test_err, xlabel="number of trees", ylabel="error"
)
ax = fig.axes[0]
ax.set_xscale("log")
ax.legend(["OOB error", "test error"])  # the helper's default labels are training/test
fig.tight_layout()
print("OOB error: ", [round(e, 3) for e in oob_err])
print("test error:", [round(e, 3) for e in test_err])

**Read the figure.** At a handful of trees the OOB error is erratic and scikit-learn warns: at 3 trees,
about a quarter of the points (0.63³ ≈ 0.25) are never out-of-bag, so the estimate rests on too little.
By around **25 trees** the warning is gone and the OOB error settles, from then on running **roughly
parallel to the test error, about one to two accuracy points below it** — consistently optimistic, not
converging onto it. Past a few dozen trees, OOB is a steady (if slightly rosy) free read on how the
forest will generalize.

## OOB or cross-validation?

Both estimate generalization without spending the test set, and they suit slightly different needs.
**Cross-validation** refits the model k times on k different splits — thorough, model-agnostic, the
right tool when you compare across model *families* (a forest vs an SVM). **OOB** is almost free: it
falls out of the single fit that trained the forest, with no refitting, which makes it handy for quick
checks while tuning a forest's `n_estimators` or `max_features`. Whichever you use to *select*, the
number you *report* still comes from the sealed test.

## Your turn

1. **Grade by hand (easy).** Four trees were trained on these bootstraps of points `A–F`: T1 saw
   `{A, B, C, C}`, T2 saw `{B, D, E, F}`, T3 saw `{A, A, D, F}`, T4 saw `{C, D, E, E}`. Which trees are
   allowed to grade point `A`? Point `E`? If `A`'s OOB graders vote `malignant, benign`, what is `A`'s
   OOB prediction (and what does the tie say)?

2. **Count the graders (medium).** For n = 1000 points and B = 100 trees, about how many trees grade
   each point? Using P(a point is in-bag in one tree) ≈ 0.63, estimate the chance a point has *no* OOB
   grader at all.

3. **Find the smallest trustworthy forest (harder).** Sweep `n_estimators` and plot OOB error and test
   error together. Where does the sklearn warning stop, and from how many trees would you trust the OOB
   estimate to choose a hyperparameter?

## What you built

- You derived the **out-of-bag fraction** `(1 − 1/n)ⁿ → 1/e ≈ 0.37` and measured it.
- You built the **OOB prediction** by hand — each point judged by the ~73 trees that never saw it — and
  read off an OOB accuracy of **0.962**.
- You matched your hand-built estimate to scikit-learn's **`oob_score_`** (0.955) and saw OOB settle one
  to two points above the **sealed test** (0.942) — free, but mildly optimistic.
- You learned **when to trust OOB**: with enough trees it is a free, reliable estimate; with too few,
  scikit-learn warns and the estimate is unreliable.

**Vocabulary you now own:** out-of-bag (OOB) · OOB error · `oob_score_` · free validation estimate.

## Going further (optional)

OOB does more than score the forest. Because each point has a set of trees that never saw it, you can
permute a feature and measure the drop in OOB accuracy — **OOB permutation importance** — a built-in,
held-out way to rank features (notebooks 4–5). And OOB is handy for cheap hyperparameter checks on large
forests. Two cautions to carry: OOB is slightly optimistic, and it is unreliable for tiny forests — for
a final, defensible number, a sealed test (or nested cross-validation) still wins.

## References

- Breiman, L. (1996). *Bagging predictors.* Machine Learning 24, 123–140 — introduces out-of-bag
  estimation. DOI: [10.1007/BF00058655](https://doi.org/10.1007/BF00058655)
- Breiman, L. (2001). *Random Forests.* Machine Learning 45, 5–32 — OOB error and importance.
  DOI: [10.1023/A:1010933404324](https://doi.org/10.1023/A:1010933404324)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §15.3.1 (out-of-bag samples). DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7)

*Previous: 02 — The "random" in the forest: decorrelating the trees. Next: 04 — The estimator and its
parameters.*